# ___ Modeling Draft 3 ___ 

**Objectif :** Comparer les modèles avec et sans features à risque de Data Leakage + Analyse Overfitting/Underfitting.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.model_selection import StratifiedKFold, cross_val_score, learning_curve
from sklearn.metrics import f1_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.svm import SVC

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 5)

---
##  Chargement des données

In [7]:
X_train = pd.read_csv('../data/processed/X_train.csv')
X_test = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
y_test = pd.read_csv('../data/processed/y_test.csv').squeeze()

print(f'Shape = Train: {X_train.shape} | Test: {X_test.shape}')

Shape = Train: (1600, 20) | Test: (400, 20)


---
## Détection des Features à Risque de Data Leakage

In [ ]:
corr = X_train.corrwith(y_train).abs().sort_values(ascending=False)
print("Top 12 Features les plus corrélées avec PCOS :")
print(corr.head(12))

leakage_features = ['Follicle No. (L)', 'Follicle No. (R)', 
                    'Avg. F size (L) (mm)', 'Avg. F size (R) (mm)']

X_train_full = X_train.copy()
X_test_full = X_test.copy()

X_train_reduced = X_train.drop(columns=leakage_features, errors='ignore')
X_test_reduced = X_test.drop(columns=leakage_features, errors='ignore')

Top 12 Features les plus corrélées avec PCOS :
Follicle No. (R)        0.634947
Follicle No. (L)        0.601352
Skin darkening (Y/N)    0.473487
hair growth(Y/N)        0.473095
Weight gain(Y/N)        0.422705
Cycle(R/I)              0.401711
Fast food (Y/N)         0.396518
AMH(ng/mL)              0.310715
Pimples(Y/N)            0.269956
BMI                     0.199799
Hair loss(Y/N)          0.187957
Age (yrs)               0.184634
dtype: float64

Features avec toutes les colonnes : 20
Features sans leakage (réaliste)   : 16


---
## Fonction d'évaluation

In [4]:
results = []

def evaluate_model(model, name, X_tr, X_te, version="Full"):
    pipeline = Pipeline([('scaler', StandardScaler()), ('clf', model)])
    pipeline.fit(X_tr, y_train)
    
    y_pred = pipeline.predict(X_te)
    y_prob = pipeline.predict_proba(X_te)[:, 1]

    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    cv_f1 = cross_val_score(pipeline, X_tr, y_train, cv=cv, scoring='f1')

    gap = abs(cv_f1.mean() - f1)

    results.append({
        'Model': name,
        'Version': version,
        'F1-Score': round(f1, 4),
        'ROC-AUC': round(auc, 4),
        'CV_F1': round(cv_f1.mean(), 4),
        'CV_Std': round(cv_f1.std(), 4),
        'Gap': round(gap, 4)
    })

    print(f'{name:20} | {version:6} → F1: {f1:.4f} | AUC: {auc:.4f} | CV: {cv_f1.mean():.4f} | Gap: {gap:.4f}')

---
## Entraînement des Modèles (Full vs Reduced)

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=5, random_state=42, eval_metric='logloss'),
    "LightGBM": LGBMClassifier(n_estimators=300, learning_rate=0.1, max_depth=5, random_state=42, verbose=-1),
    "SVM RBF": SVC(probability=True, kernel='rbf', random_state=42)
}

print("\n___ Version FULL (avec risque de leakage) ___")
for name, clf in models.items():
    evaluate_model(clf, name, X_train_full, X_test_full, "Full")

print("\n___ Version REDUCED (sans features folliculaires) ___")
for name, clf in models.items():
    evaluate_model(clf, name, X_train_reduced, X_test_reduced, "Reduced")


=== VERSION FULL (avec risque de leakage) ===
Logistic Regression  | Full   → F1: 0.8745 | AUC: 0.9621 | CV: 0.8293 | Gap: 0.0452
Random Forest        | Full   → F1: 0.9918 | AUC: 0.9999 | CV: 0.9887 | Gap: 0.0031
XGBoost              | Full   → F1: 0.9878 | AUC: 0.9999 | CV: 0.9907 | Gap: 0.0029
LightGBM             | Full   → F1: 0.9959 | AUC: 1.0000 | CV: 0.9876 | Gap: 0.0083
SVM RBF              | Full   → F1: 0.9712 | AUC: 0.9984 | CV: 0.9511 | Gap: 0.0201

=== VERSION REDUCED (sans features folliculaires) ===
Logistic Regression  | Reduced → F1: 0.8170 | AUC: 0.9069 | CV: 0.7668 | Gap: 0.0502
Random Forest        | Reduced → F1: 0.9796 | AUC: 0.9991 | CV: 0.9662 | Gap: 0.0134
XGBoost              | Reduced → F1: 0.9794 | AUC: 0.9947 | CV: 0.9792 | Gap: 0.0002
LightGBM             | Reduced → F1: 0.9878 | AUC: 0.9970 | CV: 0.9761 | Gap: 0.0117
SVM RBF              | Reduced → F1: 0.8718 | AUC: 0.9749 | CV: 0.8412 | Gap: 0.0306


---
##  Résultats Comparatifs

In [6]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by=['Version', 'F1-Score'], ascending=[False, False])
results_df

,Model,Version,F1-Score,ROC-AUC,CV_F1,CV_Std,Gap
8,LightGBM,Reduced,0.9878,0.9970,0.9761,0.0115,0.0117
6,Random Forest,Reduced,0.9796,0.9991,0.9662,0.0191,0.0134
7,XGBoost,Reduced,0.9794,0.9947,0.9792,0.0080,0.0002
9,SVM RBF,Reduced,0.8718,0.9749,0.8412,0.0473,0.0306
5,Logistic Regression,Reduced,0.8170,0.9069,0.7668,0.0489,0.0502
3,LightGBM,Full,0.9959,1.0000,0.9876,0.0089,0.0083
1,Random Forest,Full,0.9918,0.9999,0.9887,0.0070,0.0031
2,XGBoost,Full,0.9878,0.9999,0.9907,0.0072,0.0029
4,SVM RBF,Full,0.9712,0.9984,0.9511,0.0233,0.0201
0,Logistic Regression,Full,0.8745,0.9621,0.8293,0.0311,0.0452
